# A setup for AI-assisted programming

This notebook describes how you can setup a system to use agents powered by large language models (LLMs) to assist you in programming tasks.

## Configuring access to the Imperial HX3 cluster

During these workshops we will be using LLMs which will be running on College infrastructure, namely the [HX3 cluster](https://icl-rcs-user-guide.readthedocs.io/en/latest/hpc/hx3/).
There are many reasons as to why one would choose to run one's own LLM, two important ones are:

* Data safety: your data is not shared with 3<sup>rd</sup> parties. This is very important in a research context, as you want to share your findings via a publication not via an LLM which will not attribute it to you.
* No need to pay a subscription to companies potentially engaged in unethical practices.

You should already be registered on the HX3 system. You now need to create a token (also called an API key) which will allow your agent to connect to the LLM running on HX3. There are 2 steps to do this, first generate a Client_ID/Client_secret pair, and then from those generate the actual token.

### Generate Client_ID/Client_secret (Access Key)

1. First connect to HX3 Run:ai platform: [https://runai.hx3.hpc.ic.ac.uk](https://runai.hx3.hpc.ic.ac.uk/).
2. You can ignore all the information on the screen and click the avatar on the top right corner of the screen, and choose the option *Settings*.
3. Choose the option *Access Keys*, and press the link `+ Access Key`.
4. You can name this access key whatever you like. For example: ai-programming-workshop
5. On the next screen you will see the Client_ID/Client_secret pair that was generated (you will need to click on the 'eye' icon to see the client secret. Copy the Client_ID and Client_secret values to the code cell below. After you copy the value you can press `Done` and close the Run:ai webpage.

In [ ]:
runai_client_id=""
runai_client_secret=""

### Generating tokens/API key

The code cell below uses your Client_ID/Client_secret pair defined above to connect to HX3 and generate a token.

In [ ]:
import urllib.request
import urllib.parse
import json

data = {
    "grantType": "client_credentials",
    "clientId": runai_client_id,
    "clientSecret": runai_client_secret
}

req = urllib.request.Request(
    "https://runai.hx3.hpc.ic.ac.uk/api/v1/token",
    data=json.dumps(data).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

with urllib.request.urlopen(req) as response:
    response_data = json.loads(response.read().decode("utf-8"))
    
token = response_data["accessToken"]
print(token)

Tokens generated by HX3 expire after about one hour. We will use the token generated here during configuration of the agent below, but instead of repeatedly generating tokens ourselves we will configure our agent to call a script with code above and generate a new token when needed.

Edit the script *HX3_tokens.py* on the same folder as this notebook and add your Client_ID/Client_secret pair. Note that your Client_ID/Client_secret pair is sensitive information, equivalent to login/password pair, so you want to keep the script private and safe.

## Installing software

For the purpose of these workshops we will install 3 pieces of software:

**Visual Studio Code (aka VScode)**
: This is a popular integrated development environment (IDE) developed by Microsoft. One could use one of many other existing IDEs.

**VS Code Goose extension**
: This an interface between VScode and our chosen agent called Goose.

**Goose**
: This is our *harness* or agent, which passes on our queries to the LLM and relay its output. Importantly, the agent does not simply display the text output by the LLM, but it is also able to perform actions, such as creating new files, or be asked to read the context of files or websites.

### Installing Visual Studio Code

[Download](https://code.visualstudio.com/download) and run the installer for your platform.

When asked about the installation of the Copilot agent and use of a Github account it is safe to say no. We will be using a different agent.

### Installing the VS Code Goose extension

To install extensions in VScode, go to *View* on the top menu and select *Extensions*. Search for "VS Code Goose", check that the publisher is "Block", and install the extension.

### Installing and configuring Goose CLI

We now want to install the command line interface (CLI) version of Goose. We will describe the installation procedure for Windows, but if you are using a different platform, you can use the [Goose installation instructions](https://goose-docs.ai/docs/getting-started/installation) from the website.

On Windows, open the PowerShell and run the following command

    Invoke-WebRequest -Uri "https://raw.githubusercontent.com/aaif-goose/goose/main/download_cli.ps1" -OutFile "download_cli.ps1";
    
This will download the installation script `download_cli.ps1`. We now run the script by typing in the PowerShell

    .\download_cli_ps1
    
This will download and install Goose CLI files and initiate the configuration process.

The configuration is a sequence of questions and answers. We will go over them here, but if you feel impatient and just want to get started, you can interrupt the configuration by pressing the key combination `Ctrl+C`, and copy the configuration file *config.yaml* you find in the same folder as this notebook, to the folder %APPDATA%\Block\goose\config\.

#### Goose CLI configuration options

    *  Share anonymous usage data to help improve goose?
    |    Yes  / > No

Safe to say 'No'.

    *  How would you like to set up your provider?
    |    OpenRouter Login (Recommended) (Sign in with OpenRouter to automatically configure models)
    |    Tetrate Agent Router Service Login
    |  > Manual Configuration

Choose 'Manual Configuration'

    *  Which model provider should we use?
    |    NVIDIA
    |    NanoGPT
    |    Novita AI
    |    OVHcloud
    |    Ollama
    |    Ollama Cloud
    |  > OpenAI (OpenAI models, including OpenAI compatible ones)
    |    OpenCode Go
    |    OpenCode Zen
    |    OpenRouter

Scroll down the list and choose 'OpenAI (OpenAI models, including OpenAI compatible ones)'. Note that we will not be using OpenAI as a provider, but we are using a OpenAI compatible API to interact with the model on HX3.

    *  Would you like to set OPENAI_API_KEY? (optional)
    |  > Yes  /   No

Say 'Yes'.

    *  Enter value for OPENAI_API_KEY

Copy the token generated at the top of this notebook and paste it to the PowerShell using the right mouse button.

    *  Would you like to configure advanced settings?
    |  > Yes  /   No

Say 'Yes'.

    *  Enter OPENAI_BASE_URL (optional, press Enter to skip)

Safe to skip.

    *  Provider OpenAI requires OPENAI_HOST, please enter a value

You should type <span style="color:green">https://qwen-chem-tok-runai-chemistry-tests.runai-inference.hx3.hpc.ic.ac.uk</span>. (To paste on the PowerShell use the right-click button of the mouse.)

    *  Enter OPENAI_BASE_PATH (optional, press Enter to skip)

You should type `v1/chat/completions`.

    *  Enter OPENAI_ORGANIZATION (optional, press Enter to skip)

Fine to skip.

    *  Enter OPENAI_PROJECT (optional, press Enter to skip)

Skip.

    *  Would you like to set OPENAI_CUSTOM_HEADERS? (optional)
    |    Yes  / > No

Say 'No'.

    *  Enter OPENAI_TIMEOUT (optional, press Enter to skip)
    |  600

Press return to keep the default value.

Goose will the detect the model that is being offered by HX3 as:

    o  Model fetch complete
    |
    *  Select a model:
    |  > Qwen/Qwen3-0.6B
    |    Enter a model not listed...

Choose the default `Qwen/Qwen3-0.6B`.

Goose then asks an important question about how it should operate.

    *  Which goose mode would you like to configure?
    |    Auto Mode
    |  > Approve Mode (All tools, extensions and file modifications will require human approval)
    |    Smart Approve Mode
    |    Chat Mode

We should not give the agent free reign over our files and our data. It is therefore important that the agent asks for approval for modifications.

This completes the configuration. To test if the agent is correctly configured you can start a session on the command line by typing in the PowerShell

    goose session